# Amparo -- RAG avanzado + tool use

Monta las dos tecnicas de recuperacion avanzada (hybrid search y reranking)
sobre el RAG base y expone el retrieval como herramienta (function calling).
Toda la logica vive en `tools/rag/`; aca se clona el repo, se instala el stack
pesado y se corren las fases.

**Las tres configuraciones que se comparan** (lo unico que cambia entre ellas es
el retrieval; mismo prompt, mismo generador, mismo eval set):

| Config | `use_hybrid` | `use_rerank` | Retrieval |
|---|---|---|---|
| **A** | False | False | denso puro (coseno e5) |
| **B** | True | False | denso + BM25, fusion RRF |
| **C** | True | True | hybrid -> cross-encoder reordena |

**Requiere GPU** (T4 o superior): e5, el cross-encoder y Qwen2.5-7B corren aca.

Las decisiones de diseno de cada tecnica estan en `docs/m3_decisiones_rag.md`.

Cada respuesta se emite en formato Ragas (`pipeline.to_eval_record`) con los
flags `used_hybrid`/`used_rerank`, que identifican la configuracion que la
genero.

In [ ]:
# Rama del repo a clonar. Cambiala si el codigo del RAG avanzado todavia no esta
# en main (por ejemplo, mientras vive en una rama de PR).
RAMA = "main"

import os

if not os.path.isdir("Amparo"):
    !git clone -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd Amparo && git fetch origin && git checkout {RAMA} && git pull

%cd Amparo

In [ ]:
import os, sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

REQUERIDOS = [
    "tools/rag/hybrid.py",
    "tools/rag/rerank.py",
    "tools/rag/tools.py",
    "tools/rag/pipeline.py",
    "data/eval_set.json",
]
faltan = [r for r in REQUERIDOS if not os.path.exists(r)]
if faltan:
    raise SystemExit(
        f"Falta en la rama '{RAMA}': {faltan}\n\n"
        "Este notebook corre contra el repo REMOTO. Si el codigo del RAG avanzado "
        "todavia esta solo en tu maquina, commitealo y pusheralo, o cambia RAMA."
    )
print("Repo OK.")

In [ ]:
# Dependencias livianas del repo (incluye faiss-cpu y rank_bm25, el BM25 de la
# hybrid search).
!pip install -q -r requirements.txt

In [ ]:
# Stack pesado: se instala aqui y no en requirements.txt, para no reemplazar el
# build de PyTorch con CUDA que Colab ya trae. sentence-transformers trae el
# cross-encoder del reranking; transformers alcanza para e5 y la generacion;
# peft/bitsandbytes solo si se genera con el adaptador LoRA.
!pip install -q -U transformers sentence-transformers peft bitsandbytes accelerate

## Fase 1 -- Cargar el indice y construir el BM25

El indice denso (FAISS) se construye en `rag_ingenuo.ipynb` y se guarda en Drive.
Aca se carga ya hecho. El indice BM25 se construye en memoria a partir de la
metadata del store (los mismos chunks, en el mismo orden): es Python puro y
tarda segundos; se construye una vez y se reusa en toda la corrida.

In [ ]:
from google.colab import drive
import pathlib, shutil
from tools.rag import config

drive.mount("/content/drive")
origen = pathlib.Path(config.DRIVE_ROOT) / "rag"
config.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
for nombre in ("rag_index.faiss", "rag_index_metadata.jsonl"):
    shutil.copy(origen / nombre, config.ARTIFACTS_DIR / nombre)
print("Indice copiado de Drive a artifacts/.")

In [ ]:
from tools.rag import pipeline
from tools.rag.hybrid import BM25Index

store = pipeline.load_index()
bm25 = BM25Index(store.metadata)   # una sola vez; se reusa en cada consulta
print(f"Indice denso: {len(store)} chunks | BM25: {len(bm25)} chunks")

## Fase 2 -- Comparacion lado a lado: A vs B vs C sobre el retrieval

Antes de generar, se mira solo lo que recupera cada configuracion. Tres consultas
elegidas para exponer donde cada tecnica aporta:

1. **Termino exacto** (`articulo 64` del CST): donde el denso difumina y BM25
   rescata el chunk correcto -> B/C deberian subirlo.
2. **Coloquial** (lenguaje del usuario, sin terminos tecnicos): donde el denso ya
   es fuerte -> hybrid no debe empeorarlo.
3. **Fuera del corpus**: la valvula de escape debe activarse en las tres.

Solo se recupera, sin generar: es el diagnostico por consulta, mas informativo
que la metrica agregada.

In [ ]:
from tools.rag.retrieve import retrieve

SISTEMAS = {
    "A denso": dict(use_hybrid=False, use_rerank=False),
    "B +hybrid": dict(use_hybrid=True, use_rerank=False),
    "C +rerank": dict(use_hybrid=True, use_rerank=True),
}

consultas_demo = [
    "que dice el articulo 64 del codigo sustantivo del trabajo",   # termino exacto
    "me despidieron sin pagarme la liquidacion, que hago",          # coloquial
    "cual es el plazo para apelar una multa de transito en Argentina",  # fuera de corpus
]

for q in consultas_demo:
    print("=" * 78)
    print("CONSULTA:", q)
    for nombre, banderas in SISTEMAS.items():
        # min_score=None para ver que recupera cada configuracion antes de la
        # valvula de escape.
        crudos = retrieve(q, store, top_k=3, min_score=None, bm25=bm25, **banderas)
        citas = [f"{r.cita} (dense={r.dense_score:.2f})" if r.dense_score is not None else f"{r.cita} (solo BM25)" for r in crudos]
        print(f"  {nombre}: {citas}")

## Fase 3 -- Cargar el generador

Qwen2.5-7B-Instruct, cargado una sola vez y reusado. `USE_LORA=True` genera con
el adaptador LoRA encima.

In [ ]:
USE_LORA = False
model_bundle = pipeline.load_model(use_lora=USE_LORA)
print("Modelo cargado.")

## Fase 4 -- Corrida A/B/C sobre el eval set (con latencia)

Corre las tres configuraciones sobre `data/eval_set.json` y produce, por cada
una, la lista de registros en formato Ragas (`pipeline.to_eval_record`). Mide la
latencia por consulta de cada configuracion: toda tecnica cobra en tiempo, y ese
costo se reporta junto a la calidad.

Las tres corridas quedan etiquetadas por configuracion (`used_hybrid`/
`used_rerank`) y son comparables: mismo eval set, mismo generador.

In [ ]:
import json, time
from tools.evaluation import eval_set as eval_set_mod

registros = eval_set_mod.load_eval_set()
print(f"{len(registros)} registros ({len(eval_set_mod.gold_examples(registros))} gold, "
      f"{len(eval_set_mod.adversarial_examples(registros))} adversariales)")

corridas = {}   # nombre_config -> lista de eval_records
latencias = {}  # nombre_config -> seg/consulta

for nombre, banderas in SISTEMAS.items():
    print(f"\nCorriendo configuracion {nombre} ...")
    records, t0 = [], time.perf_counter()
    for reg in registros:
        consulta = reg["messages"][1]["content"]
        resultado = pipeline.answer_query(
            consulta, store, use_lora=USE_LORA, bm25=bm25,
            model_bundle=model_bundle, **banderas,
        )
        records.append(pipeline.to_eval_record(resultado, reg))
    latencias[nombre] = (time.perf_counter() - t0) / len(registros)
    corridas[nombre] = records
    print(f"  {nombre}: {latencias[nombre]:.2f} seg/consulta")

In [ ]:
# Persistir las tres corridas en Drive para levantarlas sin re-generar.
destino = pathlib.Path(config.DRIVE_ROOT) / "rag" / "corridas_abc"
destino.mkdir(parents=True, exist_ok=True)

for nombre, records in corridas.items():
    slug = nombre.split()[0].lower()   # a / b / c
    ruta = destino / f"eval_records_config_{slug}.json"
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"  {nombre} -> {ruta}")

print("\nLatencia por consulta:")
for nombre, seg in latencias.items():
    print(f"  {nombre}: {seg:.2f} s")

### Tabla de comparacion A/B/C

La calidad (scorecard del harness + RAGAS) se calcula sobre los tres archivos
`eval_records_config_{a,b,c}.json`. Estructura de la tabla:

| Config | Scorecard | faithfulness | context precision | context recall | answer relevancy | seg/consulta |
|---|---|---|---|---|---|---|
| A | | | | | | |
| B | | | | | | |
| C | | | | | | |

Lectura:
- **B > A** en context precision/recall: el corpus tiene vocabulario exacto que
  el denso difuminaba y el hybrid lo recupera.
- **C > B**: habia ruido en el top-k y el reranker lo limpia.
- Una tecnica que no mejora una metrica no ataca ese fallo concreto; se reporta
  junto a su latencia para decidir si su costo se justifica.

## Fase 5 -- Tool use: el retrieval como herramienta

En vez de recuperar siempre, el modelo decide si llamar `buscar_normas`. La
herramienta invoca el retrieval avanzado (configuracion C). Dos casos: uno que
amerita buscar (fundamentar en una norma) y uno que no (un saludo).

In [ ]:
from tools.rag import tools

for consulta in [
    "me despidieron sin justa causa, que derechos tengo",   # deberia buscar
    "hola, buenas tardes",                                    # no deberia buscar
]:
    print("=" * 78)
    print("USUARIO:", consulta)
    r = tools.responder_con_tools(consulta, store, bm25=bm25, model_bundle=model_bundle)
    for tc in r["tool_calls"]:
        print(f"  -> llamo {tc['tool']}({tc['args']})")
    print("AMPARO:", r["response"])

---
### Contenido

1. Codigo de las dos tecnicas (`tools/rag/hybrid.py`, `tools/rag/rerank.py`),
   integradas en `retrieve.py`/`pipeline.py` por bandera, y la herramienta
   (`tools/rag/tools.py`), con tests que corren sin GPU (`tests/rag/`).
2. Las tres corridas A/B/C en formato Ragas y la latencia por configuracion
   (Fase 4).
3. Las decisiones de diseno en `docs/m3_decisiones_rag.md`.

*Amparo · RAG avanzado + tool use.*